[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Your First Model &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Split `X` and `y` with `test_size=0.2` and `random_state=1`, and print how many stations went into each part.

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(len(X_train), "for training and", len(X_test), "for testing")

960 for training and 240 for testing


A fifth of 1,200 is 240, so 960 stations are left to train on.

**2.** Train a `LinearRegression` on that split, and print its R² on the test set rounded to three places.

In [3]:
model = LinearRegression().fit(X_train, y_train)

print(round(model.score(X_test, y_test), 3))

0.87


Another seed puts other stations in the test set, so the score differs a little from the worked example's 0.888.

**3.** Print the predicted and measured January means of the first five test stations from that model, side by side in a DataFrame.

In [4]:
side_by_side = pd.DataFrame({
    "predicted": model.predict(X_test.head(5)).round(1),
    "measured": y_test.head(5).to_numpy(),
}, index=X_test.index[:5])

side_by_side

,predicted,measured
636,-6.4,-7.6
683,-6.2,-8.2
1033,-6.4,-8.2
190,-1.8,0.6
1083,-1.9,-1.1


Giving the DataFrame the test stations' own index keeps each row under the label it has in `network`.

**4.** Fit `DecisionTreeRegressor(max_depth=d, random_state=0)` for `d` in 2, 4 and 8 on the split from the worked examples, and print each one's R² on the training set and on the test set.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

for depth in [2, 4, 8]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    train_score, test_score = tree.score(X_train, y_train), tree.score(X_test, y_test)
    print(f"depth {depth}: training {train_score:.3f}, test {test_score:.3f}")

depth 2: training 0.592, test 0.519
depth 4: training 0.812, test 0.734
depth 8: training 0.963, test 0.805


The training score climbs with every level, from 0.592 to 0.963. The test score climbs too, to 0.805 at depth 8, and the widening gap between the two is the tree starting to memorize.

**5.** Train a `DecisionTreeClassifier(max_depth=3, random_state=0)` to predict `june_frost` on a split with `random_state=0`, and print its accuracy and the share of test stations with no frost.

In [6]:
frost = network["june_frost"]
Xf_train, Xf_test, frost_train, frost_test = train_test_split(X, frost, test_size=0.25,
                                                              random_state=0)
classifier = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xf_train, frost_train)

print("accuracy:", round(classifier.score(Xf_test, frost_test), 3))
print("no frost:", round(1 - frost_test.mean(), 3))

accuracy: 0.96
no frost: 0.93


June frost is rare, so always guessing no frost already scores close to the tree. The **Imbalanced Classes** notebook shows what accuracy hides when one answer is rare.

---

&#8592; **Back to:** [Your First Model](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/02-your-first-model.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
